# Apache Iceberg com PySpark

Notebook de referência para configurar **Spark + Apache Iceberg** sobre o dataset **Statcast** (`data/raw/statcast_data.csv`) fornecido para o trabalho.

**Base teórica:** Iceberg é um *table format* aberto para grandes volumes analíticos: mantém metadados versionados (manifests, snapshots) e permite mutações **ACID** com SQL padrão. No experimento, comparamos este fluxo ao **Delta Lake** mantendo o mesmo motor (**PySpark**), o mesmo CSV e operações de manutenção alinhadas ao notebook `delta-lake.ipynb`: correção de **`velocidade_media`** após revisão de medição e **remoção** do registro do arremessador **Rodón, Carlos** (cenário de sanção ou dado inválido). Ao final, consultamos **snapshots** e **history** como evidência de auditoria.

## Configuração do Spark com Iceberg

**O que este bloco faz:** define o caminho do *warehouse* local (`data/iceberg_warehouse`), declara o pacote Maven do **Iceberg Spark runtime** compatível com Spark 3.5 e registra o catálogo `ice` como `SparkCatalog` em modo Hadoop. As extensões `IcebergSparkSessionExtensions` habilitam planejamento e execução de comandos Iceberg no Spark SQL. O `defaultCatalog` aponta para `ice`, de modo que tabelas qualificadas como `ice.baseball...` sejam resolvidas no armazenamento configurado.

**Conceito:** o *warehouse* concentra dados e metadados das tabelas Iceberg; cada operação DML gera um novo **snapshot** consultável posteriormente.

In [1]:
from pathlib import Path
from pyspark.sql import SparkSession

raiz_projeto = Path("..").resolve()
warehouse_path = str(raiz_projeto / "data" / "iceberg_warehouse")

ICEBERG_RUNTIME = "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.2"

spark = (
    SparkSession.builder
    .appName("Iceberg_Statcast")
    .config("spark.jars.packages", ICEBERG_RUNTIME)
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.ice", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.ice.type", "hadoop")
    .config("spark.sql.catalog.ice.warehouse", warehouse_path)
    .config("spark.sql.defaultCatalog", "ice")
    .getOrCreate()
)

print("Warehouse Iceberg:", warehouse_path)
spark.sql("SELECT current_catalog() AS catalogo, current_database() AS banco").show(truncate=False)

26/04/27 22:25:32 WARN Utils: Your hostname, PC-Octavio resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/27 22:25:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/mnt/c/source/trabalho1ed/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/octavio/.ivy2/cache
The jars for the packages stored in: /home/octavio/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-bf14eb54-eebf-4a71-a901-375e207fbeb6;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.2 in central
downloading https://repo1.maven.org/maven2/org/apache/iceberg/iceberg-spark-runtime-3.5_2.12/1.5.2/iceberg-spark-runtime-3.5_2.12-1.5.2.jar ...
	[SUCCESSFUL ] org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.2!iceberg-spark-runtime-3.5_2.12.jar (1418ms)
:: resolution report :: resolve 850ms :: artifacts dl 1422ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.2 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evict

Warehouse Iceberg: /mnt/c/source/trabalho1ed/data/iceberg_warehouse
+--------+-----+
|catalogo|banco|
+--------+-----+
|ice     |     |
+--------+-----+



## Carga inicial da tabela Iceberg (Statcast)

**O que este bloco faz:** importa `ler_e_limpar_dados` e `obter_esquema_statcast` de `src/ingestao.py`, lê o CSV em `data/raw/statcast_data.csv`, valida o número de colunas contra o esquema formal, cria o *namespace* `ice.baseball`, recria a tabela `statcast_arremessadores` no formato **Iceberg**, insere os dados a partir da *staging view* e exibe a contagem de linhas.

**Teoria:** a ingestão reutiliza o mesmo pipeline de limpeza do notebook Delta (*cast* e renomeação para o modelo de arremessadores), garantindo **paridade de dados**. O `CREATE TABLE ... USING iceberg` materializa arquivos de dados e metadados sob o *warehouse* configurado na célula anterior.

In [2]:
import sys

if str(raiz_projeto) not in sys.path:
    sys.path.insert(0, str(raiz_projeto))

from src.ingestao import obter_esquema_statcast, ler_e_limpar_dados

CAMINHO_CSV = str(raiz_projeto / "data" / "raw" / "statcast_data.csv")
TABELA = "ice.baseball.statcast_arremessadores"

spark.sql("CREATE NAMESPACE IF NOT EXISTS ice.baseball")
spark.sql(f"DROP TABLE IF EXISTS {TABELA}")

esquema_statcast = obter_esquema_statcast()
df = ler_e_limpar_dados(spark, CAMINHO_CSV)
assert len(df.columns) == len(esquema_statcast.fields)

df.createOrReplaceTempView("staging_statcast")

spark.sql(
    f"""
    CREATE TABLE {TABELA} (
      nome_jogador STRING,
      total_arremessos INT,
      velocidade_media FLOAT,
      taxa_giro INT,
      media_rebatidas_contra FLOAT,
      strikeouts INT,
      home_runs_cedidos INT,
      walks_cedidos INT
    ) USING iceberg
    """
)

spark.sql(
    f"""
    INSERT INTO {TABELA}
    SELECT * FROM staging_statcast
    """
)

spark.sql(f"SELECT COUNT(*) AS total_linhas FROM {TABELA}").show()

26/04/23 22:24:34 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+------------+
|total_linhas|
+------------+
|         873|
+------------+



## UPDATE no Iceberg (correção de `velocidade_media`)

**O que este bloco faz:** executa `UPDATE` em SQL sobre a tabela Iceberg, fixando **`velocidade_media = 89.2`** para **`Webb, Logan`** — o mesmo valor usado no notebook **`delta-lake.ipynb`** via API `DeltaTable` — e exibe a linha atualizada para conferência.

**Teoria:** alteração pontual preserva o restante das colunas e registra nova versão lógica da tabela (*snapshot*), sem regravar o arquivo CSV de origem.

In [4]:
print("Ajustando velocidade_media de Webb, Logan após revisão de medição (paridade com Delta)...")
spark.sql(
    f"""
    UPDATE {TABELA}
    SET velocidade_media = 89.2
    WHERE nome_jogador = 'Webb, Logan'
    """
)

spark.sql(
    f"""
    SELECT nome_jogador, velocidade_media, total_arremessos, strikeouts
    FROM {TABELA}
    WHERE nome_jogador = 'Webb, Logan'
    """
).show(truncate=False)

## DELETE no Iceberg (jogador banido / saneamento)

**O que este bloco faz:** aplica `DELETE` com predicado em **`nome_jogador = 'Rodón, Carlos'`**, reproduzindo o mesmo jogador removido no fluxo Delta; em seguida consulta **Webb, Logan** e **Rodón, Carlos** para mostrar que apenas Rodón foi expurgado.

**Teoria:** em Iceberg, deletes são planejados como novos snapshots que excluem linhas do conjunto visível, mantendo rastreabilidade via metadados.

In [5]:
print("Removendo estatísticas de Rodón, Carlos (paridade com Delta)...")
spark.sql(
    f"""
    DELETE FROM {TABELA}
    WHERE nome_jogador = 'Rodón, Carlos'
    """
)

spark.sql(
    f"""
    SELECT nome_jogador, velocidade_media, strikeouts
    FROM {TABELA}
    WHERE nome_jogador IN ('Webb, Logan', 'Rodón, Carlos')
    ORDER BY nome_jogador
    """
).show(truncate=False)

## Histórico de snapshots (auditoria Iceberg)

**O que este bloco faz:** interroga as tabelas de metadados **`{TABELA}.snapshots`** e **`{TABELA}.history`** para listar snapshots commitados, operações e qual versão está corrente — evidência de auditoria comparável ao `history()` do Delta.

**Teoria:** snapshots são imutáveis; *time travel* e leituras consistentes usam esses metadados para escolher quais arquivos de dados compõem a visão da tabela em um instante.

In [6]:
print("Snapshots Iceberg (commit, operação, manifests):")
spark.sql(f"SELECT * FROM {TABELA}.snapshots ORDER BY committed_at DESC").show(truncate=False)
print("Linha do tempo de snapshots correntes:")
spark.sql(f"SELECT * FROM {TABELA}.history ORDER BY made_current_at DESC").show(truncate=False)